# Multi-Agent AI Travel Planning System

This notebook implements a sophisticated two-phase travel planning system:

## Architecture Overview

### Phase 1: Interactive Intake Agent
- Conversational loop to collect user requirements
- Uses ConversationBufferWindowMemory (k=4)
- Collects: Destination, Budget, Interests, Available Time
- Outputs validated JSON

### Phase 2: Sequential Multi-Agent Pipeline
1. **Destination Agent**: Recommends places and activities
2. **Budget Agent**: Estimates costs and creates financial breakdown
3. **Itinerary Agent**: Generates day-by-day schedule
4. **Recommendation Agent**: Produces final markdown report

**Technology Stack**: LangChain, ChatGroq, Python

## 1. Import Dependencies

Import all necessary libraries for our multi-agent system.

In [14]:
import os
import json
from typing import Dict, Any, List, Union
from getpass import getpass

# LangChain imports
from langchain_groq import ChatGroq
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.schema import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field


print("✓ All dependencies imported successfully!")

✓ All dependencies imported successfully!


## 3. API Configuration

Set up the Groq API key for accessing the LLM.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if 'GROQ_API_KEY' in os.environ:
    print("✓ API key loaded from .env!")
else:
    print("❌ GROQ_API_KEY msh mawgood fel .env file!")

✓ API key loaded from .env!


## 4. Define Data Models

Create Pydantic models for structured data validation between agents.

In [15]:
# Data model for user requirements (Phase 1 output)
class TravelRequirements(BaseModel):
    destination: str = Field(description="The travel destination city or country")
    budget: str = Field(description="Total budget for the trip (e.g., '$2000', '1500 USD')")
    interests: str = Field(description="User's interests and preferences (e.g., 'beaches, food, culture')")
    time: str = Field(description="Available time for the trip (e.g., '5 days', '1 week')")

# Data model for destination recommendations (Agent 1 output)
class DestinationRecommendations(BaseModel):
    places: list = Field(description="List of recommended places to visit (as strings)")
    activities: list = Field(description="List of recommended activities (as strings)")
    highlights: str = Field(description="Key highlights of the destination")

# Data model for budget breakdown (Agent 2 output)
class BudgetBreakdown(BaseModel):
    accommodation: str = Field(description="Estimated accommodation costs")
    food: str = Field(description="Estimated food and dining costs")
    transport: str = Field(description="Estimated transportation costs")
    activities: str = Field(description="Estimated activities and entertainment costs")
    miscellaneous: str = Field(description="Estimated miscellaneous expenses")
    total: str = Field(description="Total estimated cost")
    budget_status: str = Field(description="Whether the plan fits within budget")

# Data model for itinerary (Agent 3 output)
class Itinerary(BaseModel):
    daily_schedule: list = Field(description="Day-by-day schedule with activities and timing (as strings)")
    duration: str = Field(description="Total trip duration")

print("✓ Data models defined successfully!")

✓ Data models defined successfully!


## 5. Initialize LLM

Create the ChatGroq instance that will be used across all agents.

In [4]:
# Initialize ChatGroq with the specified model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
    max_tokens=2048
)

print("✓ LLM initialized with model: openai/gpt-oss-120b")

✓ LLM initialized with model: openai/gpt-oss-120b


## Phase 1: Interactive Intake Agent

This section implements the conversational agent that collects user requirements.

### Key Features:
- Uses ConversationBufferWindowMemory (k=4) to remember recent conversation
- Friendly, conversational tone
- Validates all 4 required fields are collected
- Outputs structured JSON when complete

In [5]:
def create_intake_agent():
    """
    Creates the intake agent with conversation memory.
    This agent interacts with users to collect travel requirements.
    """
    # Initialize conversation memory with window size of 4
    memory = ConversationBufferWindowMemory(
        k=4,
        return_messages=True,
        memory_key="chat_history"
    )
    
    # Create the prompt template for the intake agent
    intake_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a friendly and enthusiastic travel assistant helping users plan their perfect trip.
        
Your goal is to collect EXACTLY 4 pieces of information:
1. **destination** - Where they want to go (city, country, or region)
2. **budget** - How much money they have for the trip (include currency)
3. **interests** - What they enjoy doing (activities, experiences, preferences)
4. **time** - How many days/weeks they have available

CONVERSATION PHASE:
- Be warm, conversational, and engaging
- Ask for missing information naturally
- If the user provides multiple pieces of information at once, acknowledge them
- Keep track of what you've already collected

COMPLETION PHASE:
Once you have ALL 4 pieces of information, output ONLY a valid JSON object in this exact format:
{{
    "destination": "value",
    "budget": "value",
    "interests": "value",
    "time": "value"
}}

Do NOT include any other text when outputting the JSON. Just the JSON object itself.
"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}")
    ])
    
    # Create the chain
    chain = intake_prompt | llm | StrOutputParser()
    
    return chain, memory

print("✓ Intake agent factory function created!")

✓ Intake agent factory function created!


In [6]:
def run_intake_conversation():
    """
    Runs the interactive intake conversation loop.
    Returns validated JSON with user requirements.
    """
    chain, memory = create_intake_agent()
    
    print("🌍 Welcome to the AI Travel Planning System!")
    print("="*60)
    print("Let's plan your perfect trip! I'll need a few details from you.\n")
    
    # Start the conversation
    initial_message = "Hello! I'm excited to help you plan an amazing trip. To get started, could you tell me where you'd like to go?"
    print(f"🤖 Assistant: {initial_message}\n")
    
    # Add initial message to memory
    memory.chat_memory.add_ai_message(initial_message)
    
    max_iterations = 15  # Prevent infinite loops
    iteration = 0
    
    while iteration < max_iterations:
        # Get user input
        user_input = input("👤 You: ").strip()
        print()  # Add blank line for readability
        
        if not user_input:
            print("🤖 Assistant: I didn't catch that. Could you please try again?\n")
            continue
        
        # Add user message to memory
        memory.chat_memory.add_user_message(user_input)
        
        # Get chat history for the prompt
        chat_history = memory.load_memory_variables({})["chat_history"]
        
        # Generate response
        response = chain.invoke({
            "input": user_input,
            "chat_history": chat_history
        })
        
        # Check if response is valid JSON (completion signal)
        try:
            requirements_json = json.loads(response.strip())
            
            # Validate all required fields are present
            required_fields = ["destination", "budget", "interests", "time"]
            if all(field in requirements_json for field in required_fields):
                print("✅ Great! I have all the information I need.\n")
                print("📋 Your Travel Requirements:")
                print(json.dumps(requirements_json, indent=2))
                print("\n" + "="*60)
                print("🚀 Starting multi-agent planning process...\n")
                return requirements_json
        except json.JSONDecodeError:
            # Not JSON yet, continue conversation or not all fields provided
            print(f"🤖 Assistant: {response}\n")
            memory.chat_memory.add_ai_message(response)
        
        iteration += 1
    
    print("⚠️ Maximum iterations reached. Please restart and provide information more clearly.")
    return None

print("✓ Intake conversation function ready!")

✓ Intake conversation function ready!


## Phase 2: Sequential Multi-Agent Pipeline

This section implements the four specialized agents that process the requirements.

### Agent 1: Destination Agent
Analyzes the destination and recommends specific places and activities.

In [21]:
def create_destination_agent():
    """
    Agent 1: Destination Agent
    Recommends places and activities based on user requirements.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a destination expert specializing in recommending places and activities.
        
Analyze the user's destination, interests, and available time to provide:
1. A list of must-visit places (5-8 specific locations) - as simple strings
2. A list of recommended activities matching their interests (5-8 activities) - as simple strings
3. Key highlights that make this destination special

Be specific and practical. Consider the time available when recommending activities.

IMPORTANT: Return places and activities as simple string lists, NOT as dictionaries or objects.
Example format:
{{
  "places": ["Great Pyramid of Giza", "Egyptian Museum", "Khan el-Khalili Bazaar"],
  "activities": ["Camel riding in the desert", "Nile River cruise"],
  "highlights": "Egypt offers..."
}}
"""),
        ("user", """User Requirements:
Destination: {destination}
Budget: {budget}
Interests: {interests}
Available Time: {time}

Provide your recommendations in the specified JSON format.""")
    ])
    
    # Use structured output with our data model
    json_parser = JsonOutputParser()
    chain = prompt | llm | json_parser
    
    return chain

print("✓ Destination Agent created!")

✓ Destination Agent created!


### Agent 2: Budget Agent
Estimates costs and creates a financial breakdown.

In [22]:
def create_budget_agent():
    """
    Agent 2: Budget Agent
    Estimates costs and creates financial breakdown.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a travel budget expert who provides realistic cost estimates.
        
Based on the destination, activities, and user's budget, estimate costs for:
1. Accommodation (per night and total)
2. Food and dining (per day and total)
3. Transportation (including flights if international, local transport)
4. Activities and entertainment (entry fees, tours, etc.)
5. Miscellaneous (souvenirs, tips, emergencies)

Calculate the total and determine if it fits within the user's budget.
If over budget, note that in budget_status and suggest where to save money.
Use the same currency as the user's budget.
"""),
        ("user", """User Requirements:
Destination: {destination}
Budget: {budget}
Available Time: {time}

Recommended Places: {places}
Recommended Activities: {activities}

Provide a detailed budget breakdown in the specified JSON format.""")
    ])
    
    json_parser = JsonOutputParser()
    chain = prompt | llm | json_parser
    
    return chain

print("✓ Budget Agent created!")

✓ Budget Agent created!


### Agent 3: Itinerary Agent
Creates a day-by-day schedule.

In [23]:
def create_itinerary_agent():
    """
    Agent 3: Itinerary Agent
    Creates a detailed day-by-day schedule.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert travel itinerary planner.
        
Create a detailed, realistic day-by-day schedule that:
1. Fits within the available time
2. Includes the recommended places and activities
3. Balances activity with rest time
4. Groups nearby locations on the same day
5. Includes approximate timing (morning, afternoon, evening)
6. Considers travel time between locations

Format each day as a string: "Day X: [Morning: activity] [Afternoon: activity] [Evening: activity]"
Be specific about locations from the recommendations.

IMPORTANT: Return daily_schedule as a list of strings, NOT as dictionaries.
Example format:
{{
  "daily_schedule": [
    "Day 1: Morning - Visit Pyramids, Afternoon - Egyptian Museum, Evening - Nile dinner cruise",
    "Day 2: Morning - Khan el-Khalili Bazaar, Afternoon - Coptic Cairo, Evening - Local restaurant"
  ],
  "duration": "5 days"
}}
"""),
        ("user", """User Requirements:
Destination: {destination}
Budget: {budget}
Interests: {interests}
Available Time: {time}

Recommended Places: {places}
Recommended Activities: {activities}
Budget Status: {budget_status}

Create a day-by-day itinerary in the specified JSON format.""")
    ])
    
    json_parser = JsonOutputParser()
    chain = prompt | llm | json_parser
    
    return chain

print("✓ Itinerary Agent created!")

✓ Itinerary Agent created!


### Agent 4: Recommendation Agent (Final Report)
Synthesizes all information into a beautiful markdown report.

In [24]:
def create_recommendation_agent():
    """
    Agent 4: Recommendation Agent
    Creates the final travel report in natural language (markdown format).
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a travel writer who creates inspiring, well-organized travel guides.
        
Synthesize all the information from previous agents into a beautiful, comprehensive travel report.

Your report should include:
1. A warm, engaging introduction
2. Overview of the destination with highlights
3. Complete day-by-day itinerary with descriptions
4. Budget breakdown with practical tips
5. Important recommendations and travel tips
6. What to pack suggestions
7. Final encouragement and best wishes

Use markdown formatting for readability:
- Headers (# ## ###) for sections
- Bold (**text**) for emphasis
- Lists for items
- Emojis for visual appeal

Make it inspiring and practical. This should be a report the user is excited to follow!
"""),
        ("user", """User Requirements:
Destination: {destination}
Budget: {budget}
Interests: {interests}
Available Time: {time}

Destination Highlights: {highlights}
Recommended Places: {places}
Recommended Activities: {activities}

Budget Breakdown:
- Accommodation: {accommodation}
- Food: {food}
- Transport: {transport}
- Activities: {activities_cost}
- Miscellaneous: {miscellaneous}
- Total: {total}
- Status: {budget_status}

Itinerary:
{daily_schedule}

Create a comprehensive, inspiring travel report in markdown format.""")
    ])
    
    # Use standard string output parser for natural language
    chain = prompt | llm | StrOutputParser()
    
    return chain

print("✓ Recommendation Agent created!")

✓ Recommendation Agent created!


In [25]:
def convert_to_string_list(data: Union[List, Any]) -> List[str]:
    """
    Helper function to convert list items to strings.
    Handles both simple strings and dictionaries that might be returned by the LLM.
    """
    if not isinstance(data, list):
        return [str(data)]
    
    result = []
    for item in data:
        if isinstance(item, dict):
            # Extract the most relevant value from dict
            if 'name' in item:
                result.append(str(item['name']))
            elif 'place' in item:
                result.append(str(item['place']))
            elif 'activity' in item:
                result.append(str(item['activity']))
            elif 'description' in item:
                result.append(str(item['description']))
            else:
                # Just use the first value or the whole dict as string
                result.append(str(list(item.values())[0]) if item else str(item))
        else:
            result.append(str(item))
    
    return result

print("✓ Helper functions defined!")

✓ Helper functions defined!


## 6. Orchestrate the Multi-Agent Pipeline

This function coordinates all agents sequentially.

In [29]:
def run_multi_agent_pipeline(requirements: Dict[str, Any]):
    """
    Orchestrates the sequential execution of all four agents.
    Each agent's output becomes input for the next agent.
    """
    print("\n" + "="*60)
    print("PHASE 2: MULTI-AGENT PLANNING PIPELINE")
    print("="*60 + "\n")
    
    # Agent 1: Destination Agent
    print("🏖️  AGENT 1: Destination Analysis...")
    destination_agent = create_destination_agent()
    destination_output = destination_agent.invoke({
        **requirements,
        "format_instructions": "Return valid JSON only."
    })
    
    # Use dictionary access consistently
    places = destination_output.get('places', [])
    activities = destination_output.get('activities', [])
    highlights = destination_output.get('highlights', '')
    
    print(f"   ✓ Recommended {len(places)} places and {len(activities)} activities\n")
    
    # Convert places and activities to string lists (handle potential dict responses)
    places_list = convert_to_string_list(places)
    activities_list = convert_to_string_list(activities)
    
    # Agent 2: Budget Agent
    print("💰 AGENT 2: Budget Planning...")
    budget_agent = create_budget_agent()
    budget_input = {
        **requirements,
        "places": str(places_list),
        "activities": str(activities_list),
        "format_instructions": "Return valid JSON only."
    }
    budget_output = budget_agent.invoke(budget_input)
    print(f"   ✓ Budget breakdown completed: {budget_output.get('total', 'N/A')}\n")
    
    # Agent 3: Itinerary Agent
    print("📅 AGENT 3: Itinerary Creation...")
    itinerary_agent = create_itinerary_agent()
    itinerary_input = {
        **requirements,
        "places": str(places_list),
        "activities": str(activities_list),
        "budget_status": budget_output.get('budget_status', ''),
        "format_instructions": "Return valid JSON only."
    }
    itinerary_output = itinerary_agent.invoke(itinerary_input)
    
    # Use dictionary access for daily_schedule
    daily_schedule = itinerary_output.get('daily_schedule', [])
    schedule_list = convert_to_string_list(daily_schedule)
    
    print(f"   ✓ {len(schedule_list)}-day itinerary created\n")
    
    # Agent 4: Recommendation Agent (Final Report)
    print("📝 AGENT 4: Generating Final Report...")
    recommendation_agent = create_recommendation_agent()
    
    # Prepare comprehensive input for final report - use .get() for all dictionary access
    report_input = {
        **requirements,
        "highlights": highlights,
        "places": ", ".join(places_list),
        "activities": ", ".join(activities_list),
        "accommodation": budget_output.get('accommodation', ''),
        "food": budget_output.get('food', ''),
        "transport": budget_output.get('transport', ''),
        "activities_cost": budget_output.get('activities', ''),
        "miscellaneous": budget_output.get('miscellaneous', ''),
        "total": budget_output.get('total', ''),
        "budget_status": budget_output.get('budget_status', ''),
        "daily_schedule": "\n".join(schedule_list)
    }
    
    final_report = recommendation_agent.invoke(report_input)
    print("   ✓ Final travel report generated\n")
    
    return final_report, {
        "destination_output": destination_output,
        "budget_output": budget_output,
        "itinerary_output": itinerary_output,
        "places_list": places_list,
        "activities_list": activities_list,
        "schedule_list": schedule_list
    }

print("✓ Multi-agent pipeline orchestrator ready!")

✓ Multi-agent pipeline orchestrator ready!


## 7. Main Execution: Complete System

Run the entire two-phase system.

In [30]:
def main():
    """
    Main execution function that runs the complete two-phase system.
    """
    # Phase 1: Interactive Intake
    requirements = run_intake_conversation()
    
    if requirements is None:
        print("❌ Failed to collect requirements. Please try again.")
        return
    
    # Phase 2: Multi-Agent Pipeline
    final_report, agent_outputs = run_multi_agent_pipeline(requirements)
    
    # Display the final report
    print("\n" + "="*60)
    print("FINAL TRAVEL PLAN")
    print("="*60 + "\n")
    print(final_report)
    print("\n" + "="*60)
    print("🎉 Your personalized travel plan is ready!")
    print("="*60)
    
    return final_report, agent_outputs

print("✓ Main execution function ready!")
print("\n" + "="*60)
print("Ready to start! Run the next cell to begin planning your trip.")
print("="*60)

✓ Main execution function ready!

Ready to start! Run the next cell to begin planning your trip.


## 8. Run the System

Execute this cell to start the interactive travel planning experience!

In [31]:
# Run the complete multi-agent travel planning system
if __name__ == "__main__":
    result = main()

🌍 Welcome to the AI Travel Planning System!
Let's plan your perfect trip! I'll need a few details from you.

🤖 Assistant: Hello! I'm excited to help you plan an amazing trip. To get started, could you tell me where you'd like to go?




✅ Great! I have all the information I need.

📋 Your Travel Requirements:
{
  "destination": "Spain",
  "budget": "2000$",
  "interests": "history, beaches",
  "time": "5 days"
}

🚀 Starting multi-agent planning process...


PHASE 2: MULTI-AGENT PLANNING PIPELINE

🏖️  AGENT 1: Destination Analysis...
   ✓ Recommended 8 places and 8 activities

💰 AGENT 2: Budget Planning...
   ✓ Budget breakdown completed: N/A

📅 AGENT 3: Itinerary Creation...
   ✓ 5-day itinerary created

📝 AGENT 4: Generating Final Report...
   ✓ Final travel report generated


FINAL TRAVEL PLAN

# ✈️ 5‑Day Spanish Adventure – History & Sun‑Kissed Beaches  
**Budget:** ≈ $2,000 | **Travel Style:** Mid‑range comfort, cultural immersion, beach‑relaxation  

---

## 1. Warm Welcome 🌞  

Welcome, fellow explorer!  
Spain is a tapestry of centuries‑old monuments, vibrant streets, and sparkling coastlines. In just five days you’ll wander the Moorish palaces of Granada, feel the echo of Roman gladiators in Mérida, sway to fl

## System Architecture Summary

### Phase 1: Interactive Intake Agent
```
User Input → ConversationBufferWindowMemory (k=4) → LLM (Conversational) 
     ↓
JSON Validation → Requirements JSON
```

### Phase 2: Sequential Agent Pipeline
```
Requirements JSON
     ↓
Agent 1 (Destination) → Places & Activities JSON
     ↓
Agent 2 (Budget) → Cost Breakdown JSON
     ↓
Agent 3 (Itinerary) → Day-by-Day Schedule JSON
     ↓
Agent 4 (Recommendation) → Final Markdown Report
```

### Key Technologies
- **LLM**: ChatGroq (openai/gpt-oss-120b)
- **Memory**: ConversationBufferWindowMemory
- **Output Parsing**: StructuredOutputParser (Pydantic models)
- **Orchestration**: LangChain LCEL chains

### Features
✅ Interactive conversation with memory  
✅ Structured data flow between agents  
✅ JSON validation and error handling  
✅ Sequential multi-agent coordination  
✅ Beautiful markdown report output  

---

**Note**: Make sure to set your `GROQ_API_KEY` before running the system!